In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

# 1. Muat Data
df = pd.read_csv('data.csv')
df = df.dropna()

texts = df['text'].astype(str).values
labels = df['label'].values

# 2. Konfigurasi Parameter
vocab_size = 5000
max_length = 50
embedding_dim = 32

# 3. Pra-pemrosesan Teks
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)
padded = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')

# 4. Pembagian Data (80% Latih, 20% Uji)
X_train, X_test, y_train, y_test = train_test_split(padded, labels, test_size=0.2, random_state=42)

# 5. Arsitektur Model LSTM
model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_length),
    LSTM(64),
    Dropout(0.5), # Regularisasi untuk menekan overfitting
    Dense(1, activation='sigmoid') # Output biner (0 atau 1)
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# 6. Pelatihan
model.fit(X_train, y_train, epochs=10, validation_split=0.1, verbose=1)

# 7. Evaluasi
y_pred_probs = model.predict(X_test)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()

print(f"Akurasi: {accuracy_score(y_test, y_pred):.4f}")
print("Laporan Klasifikasi:\n", classification_report(y_test, y_pred))

Epoch 1/10


c:\Anaconda3\envs\text_mining\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


94/94 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.4942 - loss: 0.6941 - val_accuracy: 0.4849 - val_loss: 0.6933
Epoch 2/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.4858 - loss: 0.6939 - val_accuracy: 0.5151 - val_loss: 0.6929
Epoch 3/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.5075 - loss: 0.6937 - val_accuracy: 0.4849 - val_loss: 0.6934
Epoch 4/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.4596 - loss: 0.6944 - val_accuracy: 0.4849 - val_loss: 0.6936
Epoch 5/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.5333 - loss: 0.6841 - val_accuracy: 1.0000 - val_loss: 0.0111
Epoch 6/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9978 - loss: 0.0129 - val_accuracy: 1.0000 - val_loss: 0.0011
Epoch 7/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9998 - loss: 0.0024 - val_accuracy: 1.0000 - val_loss: 5.2305e-04
Epoch 8/10
94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 1.0000 - loss: 9.4357e-04 - val_accuracy: 1.0000 - val_

In [ ]:
import pickle
# Skrip pelatihan lokal
model.save('model_lstm.keras')

# Simpan Tokenizer
with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)